In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ImageOps
import gradio as gr

C:\Users\locbo\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
num_classes = 10

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224 )),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1.0 - x),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

In [5]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(torch.load('efficient_net.pth'))
model = model.to(device)
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [6]:
def crop_and_center_digit(img):
    """
    Hàm tự động cắt khung chữ số và căn giữa.
    Yêu cầu đầu vào: img là ảnh xám (L), nền đen (0), nét trắng (255).
    """
    mask = img.point(lambda p: 255 if p > 15 else 0)
    bbox = mask.getbbox()
    
    if bbox is None:
        return img
        
    cropped_img = img.crop(bbox)
    
    width, height = cropped_img.size
    max_dim = max(width, height)
    new_size = max_dim + 4
    
    square_img = Image.new('L', (new_size, new_size), 0)
    
    paste_x = (new_size - width) // 2
    paste_y = (new_size - height) // 2
    square_img.paste(cropped_img, (paste_x, paste_y))
    square_img = square_img.resize((28, 28), Image.Resampling.BILINEAR)
    return square_img

In [7]:
def predict(img):
    result = {}
    for i in range(10):
        result[str(i)] = 0
    img = img['composite']
    if img is None:
        return result
    img = img.convert('L')
    img = crop_and_center_digit(img)
    input_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    for i in range(10):
        result[str(i)] = float(probabilities[i])
    return result

In [8]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(type="pil", label="Bảng vẽ"),
    
    outputs=gr.Label(num_top_classes=10, label="Xác suất dự đoán của EfficientNet-B0"),
    live=True, 
    
    title="Nhận Diện Chữ Số Viết Tay",
    description="Vẽ một chữ số từ 0 đến 9. Bảng bên phải sẽ liên tục cập nhật suy nghĩ của mô hình."
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
